In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")


### StateBackend


In [13]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


In [14]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b"
)

In [15]:
agent2 = create_deep_agent(
    model=model,
    backend=StateBackend(),
)

In [17]:
# -----------------------------------------------------------------------------
# 1. Create the agent — these two are equivalent
# -----------------------------------------------------------------------------
agent = create_deep_agent(model=model)

# Under the hood this is what `agent` is doing — explicit StateBackend:
agent2 = create_deep_agent(
    model=model,
    backend=StateBackend(),
)

In [18]:

# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
#    (StateBackend keeps that file inside LangGraph state)
# -----------------------------------------------------------------------------

result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})
# The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
✅ File `/notes/todo.txt` has been created with the requested content.


In [19]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='98f94ce1-1e05-467f-be4a-0324244a69e0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to create a file at /notes/todo.txt with content. We should use write_file. Let's ensure path exists? Probably /notes exists. We can just write. Use write_file.", 'tool_calls': [{'id': 'fc_3308c1cf-e1c7-4d4f-be05-79798199b0da', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 1943, 'total_tokens': 2029, 'completion_time': 0.114372275, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.099096732, 'prompt_tokens_details': None, 'queue_time': 

In [20]:

# -----------------------------------------------------------------------------
# 3. CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
# -----------------------------------------------------------------------------
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-08-21T13:34:21.413226+00:00', 'modified_at': '2026-08-21T13:34:21.413226+00:00'}


In [21]:
# -----------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# -----------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
1. Record video  
2. Edit video  
3. Upload video


### FileSystemBackend(local disk)

In [22]:

# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model=model,backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [23]:

# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
# -----------------------------------------------------------------------------
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
✅ File `/notes/todo.txt` has been created with the requested content.


In [24]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 C:\Users\j5414\OneDrive\Documents\GenerativeAI\54-DeepAgents\deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [26]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
1  1. Record video
2  2. Edit video
3  3. Upload video


### Deep Agent — StoreBackend verification
#### Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do. """

In [28]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


In [29]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
✅ The file `/notes/todo.txt` has been created with the requested content.


In [30]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
1. Record video
2. Edit video
3. Upload video
